In [ ]:
!pip install --quiet pydantic-ai nest_asyncio

### Verificación

In [ ]:
from importlib.metadata import version

try:
    import pydantic_ai
    import nest_asyncio
    print("✅ pydantic-ai instalado. Versión:", version("pydantic-ai"))
    print("📝 Anota esta versión en tus apuntes.")
except ImportError as error:
    print("❌ Algo falló en la instalación:", error)
    print("Vuelve a ejecutar la celda anterior sin --quiet para ver el error completo.")

## 3. Tu API key, guardada con seguridad

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key cargada correctamente.")
except Exception as error:
    print("❌ No se pudo leer el secreto OPENROUTER_API_KEY.")
    print("Revisa que exista con ese nombre exacto y que tenga activado")
    print("el interruptor 'Acceso desde el notebook'.")
    print("Detalle técnico:", error)

## 4. Un detalle de Colab: el bucle de eventos

In [ ]:
import nest_asyncio

nest_asyncio.apply()
print("✅ Listo: run_sync funcionará dentro de Colab.")

## 5. Tu primer agente

In [ ]:
from pydantic_ai import Agent

agente = Agent("openrouter:openai/gpt-4o-mini")

pregunta = "¿Cuántas bananas vale la Torre Eiffel?"
resultado = agente.run_sync(pregunta)

print(resultado.output)

## 6. Anatomía de un resultado

`resultado` no es solo texto: es un objeto con información útil. Veamos tres cosas:

- **`output`**: la respuesta del modelo.
- **`usage()`**: cuántos *tokens* se gastaron. Un token es un fragmento de texto (más o menos una sílaba o una palabra corta), y los proveedores cobran por tokens de entrada y de salida. Revisar `usage()` es un buen hábito para controlar costos.
- **`all_messages()`**: la conversación completa.

In [ ]:
print("Tipo de objeto:", type(resultado).__name__)
print()
print("Respuesta:", resultado.output)
print()
print("Uso de tokens:", resultado.usage)

Ahora la conversación. Es una lista de mensajes de dos tipos: **`ModelRequest`** (lo que *enviamos* al modelo) y **`ModelResponse`** (lo que *respondió*). Cada mensaje se compone de partes.

Esto parece un detalle técnico, pero será clave en el Artículo 4, cuando le demos memoria al agente.

In [ ]:
for mensaje in resultado.all_messages():
    print(type(mensaje).__name__)
    for parte in mensaje.parts:
        print("   -", type(parte).__name__, ":", str(parte.content)[:80])

## 7. El problema: el corredor improvisa

Hagamos un experimento. Le haremos **la misma pregunta tres veces** y compararemos las respuestas.

In [ ]:
for intento in range(1, 4):
    respuesta = agente.run_sync(pregunta)
    print(f"Intento {intento}: {respuesta.output[:250]}")
    print("-" * 60)

## 8. Personalidad y reglas con `system_prompt`

El `system_prompt` es el **contrato de trabajo** del agente: le dice quién es, cómo debe hablar y qué reglas seguir. Se define al crear el agente y se aplica a todas sus respuestas.

Un buen `system_prompt` suele tener estos ingredientes:

| Ingrediente | Ejemplo en nuestro corredor |
|---|---|
| **Rol** | "Eres Don Plátano, corredor estrella del Banano Exchange" |
| **Contexto** | "En este mercado todo se cotiza en bananas" |
| **Tono** | "Hablas con dramatismo de narrador de bolsa de valores" |
| **Restricciones** | "Responde en español y en máximo 4 oraciones" |
| **Cierre** | "Termina con una recomendación de inversión en bananas" |

Contratemos a Don Plátano:

In [ ]:
agente_corredor = Agent(
    "openrouter:openai/gpt-4o-mini",
    system_prompt=(
        "Eres Don Plátano, corredor estrella del Banano Exchange. "
        "En este mercado TODO se cotiza en bananas. "
        "Hablas con el dramatismo de un narrador de bolsa de valores: "
        "exclamas, usas jerga financiera y te emocionas con las alzas. "
        "Responde siempre en español y en máximo 4 oraciones. "
        "Termina cada cotización con una recomendación de inversión en bananas."
    ),
)

resultado_corredor = agente_corredor.run_sync(pregunta)

print(resultado_corredor.output)
print()
print("Uso de tokens:", resultado_corredor.usage)

Mismo modelo, misma pregunta, pero un comportamiento muy distinto. Observa también cuántos tokens gastó: las instrucciones se envían en cada llamada, así que **un `system_prompt` más largo cuesta un poco más**.

> 💡 PydanticAI también tiene un parámetro `instructions`, parecido a `system_prompt`. La diferencia importa cuando hay historial de conversación, y la retomaremos en el Artículo 4. Por ahora, `system_prompt` es suficiente.

## 9. Mismo agente, otro modelo

In [ ]:
resultado_gemini = agente_corredor.run_sync(
    pregunta,
    model="openrouter:google/gemini-2.5-flash-lite",
)

print(resultado_gemini.output)
print()
print("Uso de tokens:", resultado_gemini.usage)

## 10. Lo que todavía está mal

In [ ]:
seguimiento = agente_corredor.run_sync("¿Y si mejor vendo la mitad?")

print(seguimiento.output)

Don Plátano no tiene idea de qué mitad hablas: cada `run_sync` es una **conversación nueva**. Este es el inventario de pendientes:

| Problema | Cómo lo vimos | Se resuelve en |
|---|---|---|
| Inventa las cifras | Cada intento dio un precio distinto | Artículo 3 (herramientas) |
| Responde en texto libre, difícil de usar en un programa | Tendrías que extraer el número "a mano" | Artículo 2 (salidas estructuradas) |
| No recuerda la conversación | La pregunta de seguimiento lo dejó perdido | Artículo 4 (contexto y memoria) |

## 11. Errores comunes y cómo resolverlos

| Síntoma | Causa probable | Solución |
|---|---|---|
| Error al leer el secreto (`SecretNotFoundError`, `NotebookAccessError`) | El nombre no coincide o falta el acceso desde el notebook | Revisa el nombre `OPENROUTER_API_KEY` y el interruptor del panel Secrets |
| *"This event loop is already running"* | No se ejecutó `nest_asyncio.apply()` | Ejecuta la celda de la sección 4 |
| Error que dice que falta la API key | La variable de entorno se perdió al reiniciar la sesión | Vuelve a ejecutar la celda de la sección 3 |
| Error 401 | La key es inválida o está mal copiada | Genera una nueva en OpenRouter |
| Error 402 | Saldo insuficiente en OpenRouter | Carga saldo en tu cuenta |
| Error 400 o 404 mencionando el modelo | El nombre del modelo está mal escrito | Cópialo tal cual desde la página de modelos de OpenRouter |

Veamos el último caso en vivo. Pedimos un modelo que no existe y capturamos el error con `try/except`, para que el notebook no se detenga:

In [ ]:
try:
    agente_roto = Agent("openrouter:modelo/que-no-existe")
    agente_roto.run_sync("Hola")
except Exception as error:
    print("Error capturado:", type(error).__name__)
    print("Detalle (primeros 300 caracteres):")
    print(str(error)[:300])